# Run COMPASS survival pipeline locally

This notebook builds MRN lists for **six cohort arms** (three MRN-set
definitions -- `icd`, `vte`, `icd_or_vte` -- each with a **full** and an
**ARPI-restricted** variant), but only runs the survival pipeline
(Stage 3+) on the **three ARPI-restricted arms** -- `icd_arpi`,
`vte_arpi`, `icd_or_vte_arpi`. The full (non-ARPI) arms have no separate
notion of prediction-window start: every duration is measured from the
treatment anchor (first ARPI/taxane/radium-223 exposure), so a patient
without that anchor has all-NaN durations and is dropped by the landmark
filter regardless -- running the full arms end-to-end would just waste
compute reproducing the ARPI-restricted result. See cell 2 (`ALL_RUNS`
vs `RUNS`).

Full processing from the raw OncDRS pull runs in three steps:
1. `compile_COMPASS_cohort_data.py` builds the ICD inclusion/exclusion record
   plus the ARPI/chemo-anchored survival cohorts for all three cohort
   definitions (icd, vte, icd_or_vte -- each with a full and an
   ARPI-restricted variant), writing `prostate_icd_data.csv`, the six
   `prostate_arpi_survival_cohort*.csv` files, and the six bare-MRN
   `mrn_lists/*.csv` files used below as `--restrict-to-mrns` inputs.
2. `longitudinal_data_processing.py` reads the raw OncDRS tables, scan-filtered
   to the `icd_or_vte` union cohort -- cell 2 below (`--survival-cohort-csv
   {COHORT_MRN_FILES["icd_or_vte"]}`) explicitly couples this to the same
   full (non-ARPI) union CSV used as `--restrict-to-mrns` for the `icd_or_vte`
   arm. This union is a superset of all six cohort arms below, so no arm
   loses any patients here; narrower cohort selection remains a Stage 3
   concern.
3. `build_prediction_inputs.py`, run once per **ARPI-restricted** cohort
   arm below with `--restrict-to-mrns` pointed at the matching Stage 1
   CSV, builds that arm's landmark/feature matrices.

Run the shared setup and preprocessing cells first, then the per-cohort
section.


## Shared setup

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/data/gusev/USERS/jpconnor/code/CAIA")
SURVIVAL_DIR = PROJECT_ROOT / "COMPASS" / "survival_analysis"
DATA_PREPROCESSING_DIR = PROJECT_ROOT / "COMPASS" / "data_preprocessing"

NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/")
CONSOLIDATED_LABS = NEPC_PROJ_PATH / "consolidated_longitudinal_data.csv"
DATA = NEPC_PROJ_PATH / "longitudinal_prediction_data.csv"
SURVIVAL_OUTPUT_ROOT = NEPC_PROJ_PATH / "survival_analysis"

PYTHON = sys.executable
LANDMARKS = [0, 90]
N_FOLDS = 5
FORCE_RERUN = True

# COMPASS durations (t_lab, t_platinum, ...) are measured from the treatment
# anchor (first ARPI/taxane/radium-223 exposure = time 0), so anchor_col is
# "none" for every arm below: the landmark is a pure offset from the anchor
# with no anchor column. The six arms differ only in which patients are kept
# (--restrict-to-mrns), pointed at the matching Stage 1 survival cohort CSV
# written by compile_COMPASS_cohort_data.py.
COHORT_MRN_FILES = {
    "icd":             NEPC_PROJ_PATH / "prostate_arpi_survival_cohort.csv",
    "icd_arpi":        NEPC_PROJ_PATH / "prostate_arpi_survival_cohort_icd_arpi.csv",
    "vte":             NEPC_PROJ_PATH / "prostate_arpi_survival_cohort_vte.csv",
    "vte_arpi":        NEPC_PROJ_PATH / "prostate_arpi_survival_cohort_vte_arpi.csv",
    "icd_or_vte":      NEPC_PROJ_PATH / "prostate_arpi_survival_cohort_icd_or_vte.csv",
    "icd_or_vte_arpi": NEPC_PROJ_PATH / "prostate_arpi_survival_cohort_icd_or_vte_arpi.csv",
}
COHORT_TITLES = {
    "icd":             "ICD-C61 (full)",
    "icd_arpi":        "ICD-C61 (ARPI-restricted)",
    "vte":             "VTE-project prostate (full)",
    "vte_arpi":        "VTE-project prostate (ARPI-restricted)",
    "icd_or_vte":      "ICD-C61 union VTE-project prostate (full)",
    "icd_or_vte_arpi": "ICD-C61 union VTE-project prostate (ARPI-restricted)",
}

ALL_RUNS = [
    {
        "label": label,
        "title": COHORT_TITLES[label],
        "anchor_col": "none",
        "restrict_to_mrns": mrn_csv,
        "inputs_dir": SURVIVAL_OUTPUT_ROOT / f"prediction_inputs_{label}",
        "output_dir": SURVIVAL_OUTPUT_ROOT / f"local_runs_{label}",
    }
    for label, mrn_csv in COHORT_MRN_FILES.items()
]

# Only the ARPI-restricted arms have a proper treatment anchor for every
# patient (TREATMENT_ANCHOR_DATE non-null by construction -- see
# compile_COMPASS_cohort_data.py's *_arpi outputs). The non-ARPI ("full")
# arms have no separate anchor/prediction-window-start logic: patients
# without an anchor drug simply get all-NaN durations and are dropped by
# build_prediction_inputs.py's landmark filter (see build_prediction_inputs.py:500-504),
# so running the full arms end-to-end would silently reduce to the same
# ARPI-anchored patients anyway, just with wasted compute. RUNS below is
# restricted to the three *_arpi labels for that reason; ALL_RUNS (all six)
# is kept only so compile_COMPASS_cohort_data.py's MRN lists continue to be
# written/printed for all six cohort definitions.
RUNS = [run for run in ALL_RUNS if run["label"].endswith("_arpi")]

os.chdir(PROJECT_ROOT)
for run in ALL_RUNS:
    run["inputs_dir"].mkdir(parents=True, exist_ok=True)
    run["output_dir"].mkdir(parents=True, exist_ok=True)

for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"

print("python:            ", PYTHON)
print("cwd:               ", os.getcwd())
print("survival_dir:      ", SURVIVAL_DIR)
print("data_preprocessing:", DATA_PREPROCESSING_DIR)
print("consolidated_labs: ", CONSOLIDATED_LABS)
print("longitudinal_data: ", DATA)
for run in ALL_RUNS:
    tag = "RUN" if run in RUNS else "mrn-list-only"
    print(f"{run['label']:16s} [{tag:14s}]: inputs={run['inputs_dir']} outputs={run['output_dir']}")


## 1. Compile COMPASS cohort data

In [ ]:
!{PYTHON} {DATA_PREPROCESSING_DIR}/compile_COMPASS_cohort_data.py

## 2. Preprocess raw labs once (longitudinal_data_processing.py)

In [ ]:
!{PYTHON} {DATA_PREPROCESSING_DIR}/longitudinal_data_processing.py --survival-cohort-csv {COHORT_MRN_FILES["icd_or_vte"]}

## 3. Verify lab filtering

In [ ]:
df = pd.read_csv(CONSOLIDATED_LABS, low_memory=False)
print(f"consolidated rows: {len(df):,}")
print()

oor = df.loc[df["conversion_status"] == "out_of_physiologic_range", "collapsed_measurement"]
print("out_of_physiologic_range rows by collapsed_measurement (top 15):")
print(oor.value_counts().head(15).to_string() if not oor.empty else "  (none -- either no outliers or filter not active)")
print()

for lab, bound, unit in [("Testosterone", 2000, "ng/dL"), ("PSA", 10000, "ng/mL")]:
    sub = df.loc[
        (df["collapsed_measurement"] == lab) & df["numeric_result_standardized"].notna(),
        "numeric_result_standardized",
    ]
    if sub.empty:
        print(f"  {lab}: no standardized values present")
        continue
    over = int((sub > bound).sum())
    print(f"  {lab:13s}: n={len(sub):>6,}  max={sub.max():.2f} {unit}  #>{bound}={over}  (should be 0)")


## Shared run helpers

In [ ]:
TASKS = [
    ("univariate",   0,  "both",      "cox_agg_univariate_nobs_adjusted.csv"),
    ("univariate",   90, "both",      "cox_agg_univariate_nobs_adjusted.csv"),
    ("elastic-net",  0,  "both",      "cox_agg_multivariable_metrics.csv"),
    ("elastic-net",  90, "both",      "cox_agg_multivariable_metrics.csv"),
    ("elastic-net",  0,  "baseline",  "cox_agg_baseline_metrics.csv"),
    ("elastic-net",  90, "baseline",  "cox_agg_baseline_metrics.csv"),
    ("xgboost",      0,  "both",      "landmark_xgboost_metrics.csv"),
    ("xgboost",      90, "both",      "landmark_xgboost_metrics.csv"),
    ("xgboost",      0,  "baseline",  "landmark_xgboost_baseline_metrics.csv"),
    ("xgboost",      90, "baseline",  "landmark_xgboost_baseline_metrics.csv"),
]


def model_output_dir(model):
    return "cox" if model in ("univariate", "elastic-net") else "xgboost"


# Output subdir for the shared-canonical-labs univariate arm. This arm is a
# SINGLE invocation over ALL landmarks (univariate_analysis.py --shared-canonical-labs
# --landmark-days <all>): the runner intersects each landmark's canonical labs and
# tests that one shared set at every landmark, so +0d and +90d see an identical
# feature list and their per-lab HRs are directly comparable. Results land in
# cox/landmark_shared/ with landmark_days as a column inside the CSV.
SHARED_UNIVARIATE_DIR = "cox/landmark_shared"
SHARED_UNIVARIATE_FILE = "cox_agg_univariate_nobs_adjusted.csv"


def clear_prediction_inputs(inputs_dir):
    for pattern in (
        "aggregated_landmark*.csv",
        "pre_treatment_lab_long_landmark*.csv",
    ):
        for p in inputs_dir.glob(pattern):
            p.unlink()
            print(f"  removed {p.name}")
    for fname in (
        "canonical_labs_train_val.csv",
        "build_manifest.json",
        "split_assignments.csv",
        "landmark_mrn_availability.csv",
        "landmark_attrition.json",
    ):
        p = inputs_dir / fname
        if p.exists():
            p.unlink()
            print(f"  removed {p.name}")


def build_prediction_inputs(run):
    print(f"\n========== build inputs: {run['title']} ==========")
    clear_prediction_inputs(run["inputs_dir"])
    cmd = [
        PYTHON, str(DATA_PREPROCESSING_DIR / "build_prediction_inputs.py"),
        "--data", str(DATA),
        "--output-dir", str(run["inputs_dir"]),
        "--anchor-col", run["anchor_col"],
        "--landmark-days", *[str(lm) for lm in LANDMARKS],
        "--time-unit-days", "7",
        "--test-frac", "0.20",
        "--val-frac", "0.20",
        "--min-patient-coverage", "0.20",
    ]
    if run.get("restrict_to_mrns"):
        cmd += ["--restrict-to-mrns", str(run["restrict_to_mrns"])]
    print("[run ] " + " ".join(cmd))
    rc = subprocess.call(cmd)
    if rc != 0:
        raise RuntimeError(f"build_prediction_inputs failed for {run['label']} with rc={rc}")


def cohort_diagnostics(run):
    print(f"\n========== cohort diagnostics: {run['title']} ==========")
    for lm in LANDMARKS:
        agg_path = run["inputs_dir"] / f"aggregated_landmark{lm}.csv"
        if not agg_path.exists():
            print(f"  landmark +{lm}d: aggregated CSV not found, skipping")
            continue
        agg = pd.read_csv(agg_path)

        def find_col(substr, stat):
            return next(
                (c for c in agg.columns if substr.lower() in c.lower() and c.endswith(f"__{stat}")),
                None,
            )

        n_plat = int(agg["PLATINUM"].sum())
        print(f"=== landmark +{lm}d | n_total={len(agg):,} n_PLATINUM={n_plat} ===")
        for lab_substr in ("Testosterone", "PSA", "Prostate specific Ag"):
            for stat in ("mean", "last", "max", "min"):
                col = find_col(lab_substr, stat)
                if col is None:
                    continue
                for ev in (0, 1):
                    sub = agg.loc[agg["PLATINUM"] == ev, col].dropna()
                    if sub.empty:
                        continue
                    print(
                        f"  {lab_substr:>22s} {stat:5s} PLAT={ev}: median={sub.median():>10.2f} "
                        f"max={sub.max():>12.2f} n={len(sub):>5}"
                    )
                break
        print()


def build_model_command(model, landmark, config_dir, row_output_dir, run):
    if model == "univariate":
        return [
            PYTHON, str(SURVIVAL_DIR / "univariate_analysis.py"),
            "--inputs-dir", str(run["inputs_dir"]),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum",
        ]
    if model == "elastic-net":
        cmd = [
            PYTHON, str(SURVIVAL_DIR / "multivariate_analysis.py"),
            "--model", "elastic-net",
            "--inputs-dir", str(run["inputs_dir"]),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum",
            "--n-folds", str(N_FOLDS),
        ]
        if config_dir == "baseline":
            cmd.append("--baseline")
        return cmd
    if model == "xgboost":
        cmd = [
            PYTHON, str(SURVIVAL_DIR / "multivariate_analysis.py"),
            "--model", "xgboost",
            "--inputs-dir", str(run["inputs_dir"]),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum",
            "--n-folds", str(N_FOLDS),
        ]
        if config_dir == "baseline":
            cmd.append("--baseline")
        return cmd
    raise ValueError(f"Unknown model: {model}")


def run_shared_univariate(run):
    """Univariate arm on ONE shared canonical lab set across all landmarks.

    Single invocation over every landmark in LANDMARKS with
    --shared-canonical-labs, so the runner intersects each landmark's canonical
    labs and tests that shared set at every landmark. The resulting CSV carries a
    landmark_days column, so +0d and +90d rows are directly comparable per lab.
    """
    row_output_dir = run["output_dir"] / SHARED_UNIVARIATE_DIR
    row_output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = row_output_dir / SHARED_UNIVARIATE_FILE
    tag = f"{run['label']:16s} univariate  landmark_shared (labs={','.join(map(str, LANDMARKS))})"
    if metrics_path.exists() and not FORCE_RERUN:
        print(f"[skip] {tag} -> exists")
        return (tag, "skipped", 0.0)
    cmd = [
        PYTHON, str(SURVIVAL_DIR / "univariate_analysis.py"),
        "--inputs-dir", str(run["inputs_dir"]),
        "--output-dir", str(row_output_dir),
        "--landmark-days", *[str(lm) for lm in LANDMARKS],
        "--endpoints", "platinum",
        "--shared-canonical-labs",
    ]
    print(f"[run ] {tag}")
    print("       " + " ".join(cmd))
    t0 = time.time()
    rc = subprocess.call(cmd)
    elapsed = time.time() - t0
    status = "ok" if rc == 0 else f"FAILED (rc={rc})"
    print(f"[done] {tag} -> {status} ({elapsed/60:.1f} min)\n")
    return (tag, status, elapsed)


def run_models(run):
    print(f"\n========== run models: {run['title']} ==========")
    summary = []
    for model, landmark, config_dir, metrics_filename in TASKS:
        row_output_dir = run["output_dir"] / model_output_dir(model) / f"landmark_{landmark}" / config_dir
        metrics_path = row_output_dir / metrics_filename
        tag = f"{run['label']:16s} {model:11s} landmark_{landmark:<2} {config_dir}"
        if metrics_path.exists() and not FORCE_RERUN:
            print(f"[skip] {tag} -> {metrics_path.relative_to(run['output_dir'])} exists")
            summary.append((tag, "skipped", 0.0))
            continue
        row_output_dir.mkdir(parents=True, exist_ok=True)
        cmd = build_model_command(model, landmark, config_dir, row_output_dir, run)
        print(f"[run ] {tag}")
        print("       " + " ".join(cmd))
        t0 = time.time()
        rc = subprocess.call(cmd)
        elapsed = time.time() - t0
        status = "ok" if rc == 0 else f"FAILED (rc={rc})"
        print(f"[done] {tag} -> {status} ({elapsed/60:.1f} min)\n")
        summary.append((tag, status, elapsed))
    # Shared-canonical-labs univariate arm (single run over all landmarks).
    summary.append(run_shared_univariate(run))
    print("\n=== run summary ===")
    for tag, status, elapsed in summary:
        print(f"  {tag} {status:>20s} {elapsed/60:6.1f} min")
    return summary


def summarize_outputs(run):
    rows = []
    for model, landmark, config_dir, metrics_filename in TASKS:
        if model == "univariate":
            continue
        metrics_path = run["output_dir"] / model_output_dir(model) / f"landmark_{landmark}" / config_dir / metrics_filename
        base = {"run": run["label"], "model": model, "landmark": landmark, "config": config_dir, "endpoint": "platinum"}
        if not metrics_path.exists():
            rows.append({**base, "n_test": None, "n_test_events": None, "c_index": None,
                         "mean_auc_t": None, "integrated_brier": None, "status": "missing"})
            continue
        df = pd.read_csv(metrics_path)
        platinum = df.loc[df["endpoint"] == "platinum"]
        if platinum.empty:
            rows.append({**base, "n_test": None, "n_test_events": None, "c_index": None,
                         "mean_auc_t": None, "integrated_brier": None, "status": "no platinum row"})
            continue
        platinum = platinum.iloc[0]
        if model == "elastic-net":
            rows.append({
                **base,
                "n_test": int(platinum["n_test"]),
                "n_test_events": int(platinum["n_events_test"]),
                "c_index": float(platinum["test_c_index"]),
                "mean_auc_t": float(platinum["test_mean_auc_t"]),
                "integrated_brier": float(platinum["test_integrated_brier"]),
                "status": "ok",
            })
        elif model == "xgboost":
            rows.append({
                **base,
                "n_test": int(platinum["n_test"]),
                "n_test_events": int(platinum["n_test_events"]),
                "c_index": float(platinum["c_index"]),
                "mean_auc_t": float(platinum["mean_auc_t"]),
                "integrated_brier": float(platinum["integrated_brier"]),
                "status": "ok",
            })
    return pd.DataFrame(rows).sort_values(["run", "landmark", "model", "config"]).reset_index(drop=True)


def summarize_univariate_shared(run):
    """Per-lab platinum associations at +0d vs +90d on the SHARED lab set.

    Reads the shared-arm univariate CSV (identical canonical lab set at every
    landmark, landmark_days as a column) so PSA/testosterone HRs can be read
    across landmarks side by side.
    """
    path = run["output_dir"] / SHARED_UNIVARIATE_DIR / SHARED_UNIVARIATE_FILE
    if not path.exists():
        print(f"  shared univariate output missing: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    df = df.loc[df["endpoint"] == "platinum"].copy()
    keep = ["landmark_days", "feature", "lab_name", "feature_stat",
            "hazard_ratio_per_sd", "p_value", "q_value", "n_patients_used", "n_events_used"]
    keep = [c for c in keep if c in df.columns]
    sort_cols = [c for c in ("lab_name", "feature_stat", "landmark_days") if c in df.columns]
    return df[keep].sort_values(sort_cols).reset_index(drop=True)

## 4. Run the three ARPI-restricted cohort arms

Loops `build_prediction_inputs` -> `cohort_diagnostics` -> `run_models` ->
`summarize_outputs` over every entry in `RUNS` (icd_arpi, vte_arpi,
icd_or_vte_arpi only -- the full, non-ARPI-restricted arms are skipped here
since they have no proper treatment anchor for every patient; see cell 2).
Each arm is fully independent (its own `inputs_dir`/`output_dir`), so a
failure in one arm doesn't block the others from being inspected -- run
summaries are collected into `run_summaries` and per-arm metric tables into
`summary_dfs`, keyed by label.


In [ ]:
run_summaries = {}
summary_dfs = {}

for run in RUNS:
    build_prediction_inputs(run)
    cohort_diagnostics(run)
    run_summaries[run["label"]] = run_models(run)
    summary_dfs[run["label"]] = summarize_outputs(run)

summary_dfs[RUNS[0]["label"]]


### Per-cohort summary tables

Index into `summary_dfs["<label>"]` for any single arm, e.g.
`summary_dfs["icd_arpi"]` for the primary ARPI-anchored ICD-C61 cohort or
`summary_dfs["vte_arpi"]` for the ARPI-restricted VTE-project cohort. Only
the three `*_arpi` labels are populated (see cell 2 / section 4 above).


## 5. Summary


In [ ]:
combined_summary_df = pd.concat(summary_dfs.values(), ignore_index=True)
combined_summary_df
